In [41]:
import random
import numpy as np
import math 
from Strategy import Optimal , WinProb , Human , Announce , Threshold , Myopic  
from flip7_game import Game, Player ,LiveGame

In [45]:
# =================================================================
# CONFIGURATION DE L'ESPACE D'ÉTAT (MACRO-MDP)
# =================================================================
RSUP = 79               # Borne supérieure stricte de l'incrément de score sur une manche (somme + bonus Flip 7)
WIN_SCORE = 200         # Seuil critique d'arrêt (Condition de victoire)
WIN = WIN_SCORE
EXT = WIN + RSUP - 1    # Grille étendue pour absorber les débordements terminaux sans effet de bord


# =================================================================
# 1. ESTIMATION MONTE CARLO DE LA DISTRIBUTION DES INCRÉMENTS
# =================================================================

def estimate_fX(N_games=2000, seed=0):
    """
    Estime par simulation la fonction de masse (PMF) des incréments de score par manche.
    
    Hypothèse de découplage macro : On suppose la distribution des gains stationnaire
    et indépendante de l'état exact du deck afin de réduire la dimensionnalité 
    de l'espace d'état global (approximation i.i.d. des manches).
    """
    rng = random.Random(seed)
    hist = np.zeros(RSUP)
    
    for _ in range(N_games):
        # Simulation d'une dynamique de jeu standard entre 3 agents optimaux
        P = [Player(c, Optimal()) for c in "ABC"]
        g = Game(P, seed=rng.randrange(10**9))
        start = g.rng.randrange(3)
        
        for _r in range(80):
            before = [p.score for p in P]
            g.play_round(start)
            start = (start + 1) % 3
            
            # Collecte empirique des incréments de score valides
            for p, b in zip(P, before):
                inc = p.score - b
                if 0 <= inc < RSUP: 
                    hist[inc] += 1
            if any(p.score >= WIN_SCORE for p in P): 
                break
                
    return hist / hist.sum()  # Normalisation pour obtenir la loi de probabilité (PMF)

# Distribution de référence pour le calcul du point fixe
fX = estimate_fX(2000)


# =================================================================
# 2. OPÉRATEURS DE TRANSITION ET RÉSOLUTION PAR VALUE ITERATION
# =================================================================

def fwd_corr(U, f, axis):
    """
    Opérateur de corrélation croisée 'en avant' (forward cross-correlation) pondéré par f.
    Évalue vectoriellement l'espérance mathématique E[U(s + X)] où X ~ f le long de l'axe spécifié.
    Formule : out[i] = sum_x f[x] * U[i + x]
    """
    out = np.zeros_like(U)
    Nx = U.shape[axis]
    for x in range(len(f)):
        if f[x] == 0.0: 
            continue
        so = [slice(None)] * U.ndim
        si = [slice(None)] * U.ndim
        so[axis] = slice(0, Nx - x)
        si[axis] = slice(x, Nx)
        out[tuple(so)] += f[x] * U[tuple(si)]
    return out

def terminal_baseline(f1, f2, f3):
    """
    Calcule le tenseur de base terminal B[a, b, c].
    
    B[a, b, c] représente la probabilité immédiate que le joueur 1 (a) remporte la partie 
    au cours de la manche courante, sachant les scores de départ (a, b, c).
    Intègre la gestion stricte des ex-æquo (règle du partage de la victoire en cas de simultanéité).
    """
    T = np.zeros((EXT + 1, EXT + 1, EXT + 1), dtype=np.float32)
    b = np.arange(EXT + 1)
    c = np.arange(EXT + 1)
    B2, C2 = np.meshgrid(b, c, indexing='ij')
    maxBC = np.maximum(B2, C2)
    
    # Identification des états absorbants et calcul des probabilités de gain aux limites
    for a in range(EXT + 1):
        mx = np.maximum(maxBC, a)
        if not (mx >= WIN).any(): 
            continue
        is1 = (a == mx)
        # Cardinal du sous-ensemble des vainqueurs simultanés pour diviser la probabilité
        k = is1.astype(np.float32) + (B2 == mx) + (C2 == mx)
        T[a] = np.where((mx >= WIN) & is1, 1.0 / np.maximum(k, 1.0), 0.0)
        
    # Projection des conditions aux limites par application successive de l'opérateur de transition
    B = fwd_corr(fwd_corr(fwd_corr(T, f1, 0), f2, 1), f3, 2)
    return np.ascontiguousarray(B[:WIN, :WIN, :WIN])

def solve_macro(f1, f2, f3, sweeps=40, tol=2e-5):
    """
    Routage macro-markovien résolu par l'algorithme d'Itération sur la Valeur (Value Iteration).
    
    Calcule le point fixe asymptotique sous l'équation d'optimalité de Bellman.
    Le tenseur W converge vers la probabilité de gain stationnaire à long terme.
    """
    f1, f2, f3 = [np.asarray(f, dtype=np.float32) for f in (f1, f2, f3)]
    B = terminal_baseline(f1, f2, f3)
    W = B.copy()
    
    for it in range(sweeps):
        # Évaluation de l'espérance de continuation sur la grille résiduelle (W = 0 hors-grille)
        cont = fwd_corr(fwd_corr(fwd_corr(W, f1, 0), f2, 1), f3, 2)
        Wn = B + cont
        
        # Vérification du critère de convergence via la norme de Chebychev (L-infini)
        d = float(np.max(np.abs(Wn - W)))
        W = Wn
        if d < tol:
            print(f"Convergence Bellman atteinte en {it} itérations.")
            break
    return W


# =================================================================
# 3. INVOCATION ET GÉNÉRATION DE LA FONCTION DE VALEUR GLOBALE
# =================================================================

# Tenseur de probabilités stationnaires W[a, b, c]
# Représente V(a, b, c) : la probabilité de victoire finale du Joueur 1 
# à partir de n'importe quel triplet de scores cumulés courants.
W = solve_macro(fX, fX, fX)

Convergence Bellman atteinte en 14 itérations.


In [48]:
# jouons une partie
game = Game([Player("Optimal", Optimal()), Player("WinProb", WinProb(W, fX)), Player("Myopic", Myopic())])
game.play() #<------ ceci renvoi l'indice du gagnant donc 0 si Optimal gagne, 1 si WinProb gagne, 2 si Myopic gagne



1

In [42]:
# Jeux interactif entre humain et IA

# deja on commence par choisir le nom du joueur humain
human_name = input("Enter your name: ")

# La fonction qui lance le jeu
def play_live(opponents, human_name, seed=None, show_hint=True):
    """Lance un match complet : toi + les stratégies listées dans `opponents`.

    opponents : liste de (nom, classe_ou_factory_sans_argument)
                ex. [("Optimal", Optimal), ("Myopic", Myopic)]
    seed      : fixe la graine pour pouvoir rejouer exactement la même partie.
    show_hint : affiche ou non le risque de bust calculé sur le deck visible.

    Renvoie le nom du vainqueur.
    """
    human = Player(human_name, Human(human_name, show_hint=show_hint))
    others = [Player(name, Announce(name, cls())) for name, cls in opponents]
    players = [human] + others
    random.Random(seed).shuffle(players)        # ordre de table mélangé, comme en vrai

    print("Table :", ", ".join(p.name for p in players))
    game = LiveGame(players, seed=seed)
    winner_idx = game.play()

    print("\n================ RÉSULTAT ================")
    for p in players:
        marker = "  <-- VAINQUEUR" if p is players[winner_idx] else ""
        print(f"  {p.name:10s} {p.score:3d} pts{marker}")
    return players[winner_idx].name

In [44]:
# Bon maintenant vous pouvez jouer avec les IA
#play_live([("Optimal", Optimal), ("WinProb", lambda: WinProb(W, fX))])

play_live([("Optimal", Optimal), ("Raire", Optimal)],human_name)

# Bon match !

Table : Optimal, Raire, HMD

========== Manche 1 ==========

--- HMD, à toi de jouer ---
Ta main          : [(vide)]  (banque si arrêt : 0 pts)
Ton score total  : 0
  Optimal    score=  0  main=[(vide)]
  Raire      score=  0  main=[(vide)]
Deck restant     : 0:1 1:1 2:2 3:3 4:4 5:5 6:6 7:7 8:8 9:9 10:10 11:11 12:12  |  SC restantes: 3
  (indice : risque de bust au prochain tirage = 0%)
   -> HMD pioche 11, nouvelle carte.
Optimal    tire       main=[(vide)]  score=0
   -> Optimal pioche 7, nouvelle carte.
Raire      tire       main=[(vide)]  score=0
   -> Raire pioche 6, nouvelle carte.

--- HMD, à toi de jouer ---
Ta main          : [11]  (banque si arrêt : 11 pts)
Ton score total  : 0
  Optimal    score=  0  main=[7]
  Raire      score=  0  main=[6]
Deck restant     : 0:1 1:1 2:2 3:3 4:4 5:5 6:5 7:6 8:8 9:9 10:10 11:10 12:12  |  SC restantes: 3
  (indice : risque de bust au prochain tirage = 13%)
   -> HMD pioche 10, nouvelle carte.
Optimal    tire       main=[7]  score=0
   -> Opti

'Optimal'